<a href="https://colab.research.google.com/github/jacielefreitas63-tech/analisr-dados-ecommerce/blob/main/1_limpeza_e_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Projeto Delivery: Engenharia de Dados & Análise Exploratória (EDA)
> *Etapa 1:* Configuração do Ambiente e Conexão com o Data Lake (Google Drive)

Neste notebook, realizaremos a ingestão, limpeza fina, tratamento de outliers e união das bases transacionais da *Olist, malha rodoviária do **OpenStreetMap* e dados meteorológicos do *INMET*.

In [1]:
# Importação das bibliotecas fundamentais para manipulação e análise estatística
import pandas as pd
import numpy as np

print("🚀 Bibliotecas carregadas com sucesso!")

🚀 Bibliotecas carregadas com sucesso!


In [2]:
# Conexão segura com o armazenamento de dados do Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📦 1. Ingestão da Base de Pedidos (Olist Orders)
> O objetivo desta etapa é ler o dataset transacional de ordens de serviço, inspecionar os tipos de dados primitivos e mapear o volume inicial de registros antes de aplicar as regras de limpeza.

In [6]:
# Definir o caminho do arquivo no seu Drive (ajustado com _)
caminho_orders = "/content/drive/MyDrive/projeto_delivery_logistica/data/olist_orders_dataset.csv"

# Carregar o arquivo usando o Pandas
df_orders = pd.read_csv(caminho_orders)

# Exibir a volumetria do dataset (Linhas, Colunas)
print(f"Dimensões do Dataset: {df_orders.shape[0]} linhas e {df_orders.shape[1]} colunas.\n")

# Mostrar as primeiras 5 linhas para inspeção visual
df_orders.head()

Dimensões do Dataset: 99441 linhas e 8 colunas.



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
# Verificar os tipos de dados de cada coluna e a presença de valores nulos
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## 🛠️ 2. Conversão de Tipos (Cast) e Tratamento de Datas
> As colunas temporais de carimbo de data/hora (timestamps) precisam ser convertidas de object para datetime64. Isso nos permitirá calcular intervalos de tempo, como o tempo de entrega real (SLA) e atrasos.

In [8]:
# Lista de colunas que contêm datas no dataset
colunas_datas = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Converter todas as colunas da lista para o formato datetime
for coluna in colunas_datas:
    df_orders[coluna] = pd.to_datetime(df_orders[coluna])

# Validar se a conversão funcionou checando os novos tipos
df_orders[colunas_datas].dtypes

,0
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


## 🔍 3. Análise de Valores Ausentes (Missing Values)
> Antes de qualquer modelagem preditiva, mapeamos o volume absoluto e percentual de nulos para decidir entre estratégias de imputação ou remoção de registros.

In [9]:
# Calcular o total de nulos por coluna
total_nulos = df_orders.isnull().sum()

# Calcular a porcentagem de nulos por coluna
porcentagem_nulos = (df_orders.isnull().sum() / len(df_orders)) * 100

# Juntar as informações em um DataFrame de diagnóstico para o relatório
diagnostico_nulos = pd.DataFrame({'Total Nulos': total_nulos, 'Porcentagem (%)': porcentagem_nulos})
diagnostico_nulos.sort_values(by='Total Nulos', ascending=False)

,Total Nulos,Porcentagem (%)
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899
order_id,0,0.000000
order_purchase_timestamp,0,0.000000
order_status,0,0.000000
customer_id,0,0.000000
order_estimated_delivery_date,0,0.000000


## 🚚 4. Engenharia de Atributos (Feature Engineering): Métricas de SLA
> Criaremos as variáveis preditivas e de performance logística:
> 1. **tempo_entrega_dias**: Tempo real gasto desde a compra até a chegada ao cliente.
> 2. **atraso_entrega_dias**: Dias de atraso caso a entrega passe do prazo estimado (valores positivos indicam quebra de SLA).

In [10]:
# 1. Calcular o tempo de entrega real em dias
df_orders['tempo_entrega_dias'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.total_seconds() / 86400

# 2. Calcular o atraso em relação à estimativa (Valores > 0 significam atraso)
df_orders['atraso_entrega_dias'] = (df_orders['order_delivered_customer_date'] - df_orders['order_estimated_delivery_date']).dt.total_seconds() / 86400

# Substituir valores negativos por 0 no atraso (pois entregas antes do prazo não têm atraso)
df_orders['atraso_entrega_dias'] = df_orders['atraso_entrega_dias'].clip(lower=0)

# Exibir uma amostragem com as novas colunas calculadas
df_orders[['order_id', 'tempo_entrega_dias', 'atraso_entrega_dias']].head()

,order_id,tempo_entrega_dias,atraso_entrega_dias
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,0.0
